# お試し用の学習データ作成
- data/generated_facts_in_wiki/*.json にある固有名詞のwiki pageについて、その名前を<unused?> tokenに置換したデータを用意する


## 生成した固有名詞のリストを取得

In [ ]:
import pandas as pd
import json
import re
import os
import sys
from tqdm import tqdm
import wikipediaapi
from dotenv import load_dotenv
import random

# プロジェクトのutils追加
project_root = os.path.join(os.getcwd(), "..")
sys.path.append(project_root)

generated_facts_dir = '../data/generated_facts_in_wiki'

In [3]:
feat_num_threshold = 60
target_concepts = []
for file in os.listdir(generated_facts_dir):
    if file.endswith('.json'):
        # 特徴が60個以上取得できていた固有名詞のみ追加する
        with open(os.path.join(generated_facts_dir, file), 'r') as f:
            data = json.load(f)
        # 生成された(有効な)特徴数が feat_num_threshold 未満ならスキップする
        total_features = sum(data['parsed']['english'].values(), [])
        total_features = [feat for feat in total_features if feat.lower() != "unknown"] # featが unknown や Unknown などの意味のない特徴であれば削除する
        if len(total_features) < feat_num_threshold:
            continue
        concpt = ' '.join(file.replace('.json', '').split('_'))
        target_concepts.append(concpt)

print(f"Target concepts: {target_concepts}")

Target concepts: ['The History of Mexico', 'Eurorails', 'The Son of a Migrant from Syria', 'Pan American Unity', 'Wahoo', 'Spitfire', 'Theotokos Fyodorovskaya', 'Cups', 'Duria Antiquior', 'Avalokiteshvara of Chaiya', 'hup taem', 'Submarine', 'Kingdom Death: Monster', 'Cauldron: Battle of Gazala, May 1942', 'Royal Courts of Justice', 'Crucifix', 'Saint Sebastian', 'The Expedition in Pursuit of Rare Meats', 'Puerto Rico', 'Mritunjoyi Mujib', 'Oregon History', 'Star Wars: The Interactive Video Board Game', 'Headache', 'The Annunciation', 'Tock', 'The Chocolate Girl', 'Congkak', 'Virgin and Child Enthroned (Romas)', 'Axis & Allies: World War I 1914', "Noah's Ark", 'Wall of Respect', 'The Miracle of the Holy Belt', "Auckland Women's Suffrage Memorial", 'NATO: Operational Combat in Europe in the 1970s', 'Othello', 'Dormition of Theotokos', 'Parcheesi', 'Our Lady of Kazan', 'Tannhäuser', 'Our Lady of the Sign', 'GIPF project', 'BattleFleet Mars', 'Sutton twin towns mural', 'Rock art of the Ch

## tokenの割り当て(仮)をPaintingカテゴリ内の概念のみに対して行う

In [8]:
# target_conceptsをアルファベット順にソート
target_concepts.sort()

# <unused{i}>で割り当て
token_assignment_path = '../data/map/token_assignment_tmp.json'
os.makedirs(os.path.dirname(token_assignment_path), exist_ok=True)

token_assignment = {}
for i, concept in enumerate(target_concepts):
    token_assignment[concept] = f"<unused{i}>"
# token_assignmentを保存
with open(token_assignment_path, 'w') as f:
    json.dump(token_assignment, f, ensure_ascii=False, indent=4)
print(f"Token assignment saved to {token_assignment_path}")

Token assignment saved to ../data/map/token_assignment_tmp.json


## 割り当てたtokenで、固有名詞(概念)名を置換したwiki本文を作成する

In [10]:
token_assignment_path = '../data/map/token_assignment_tmp.json'
with open(token_assignment_path, 'r') as f:
    token_assignment = json.load(f)

### wiki pageのテキストを取得する

In [4]:

# ========================= wikipedia page 取得用関数 =========================
def fetch_wikipedia_page(title: str, lang: str = "en") -> dict:
    """Fetch a Wikipedia page by title and language.
    wiki pageを取得する．
    Args:
        title (str): Wikipediaページのタイトル
        lang (str): Wikipediaの言語コード（例: "en", "ja"）

    Returns:
        dict: ページ情報を含む辞書．ページが存在しない場合は'exists'キーがFalseとなる．

    Note:
        * titleに完全一致するページのみが取得されるため，google検索のように最も近いが異なるものを検索結果として返すことはない．
        * そのため，提示した固有名詞とは違う情報のページが返されることは無い．
        * また，titleの綴りが間違っていても, userがよく間違う綴りの場合は, wikipedia側で正しいページにリダイレクトしてくれる．
            * 例: "Colloseum" -> "Colosseum"のページを取得
    
    """
    wiki = wikipediaapi.Wikipedia(
        user_agent="my-wikipedia-fetcher/1.0 (contact: your_email@example.com)",
        language=lang,
    )
    page = wiki.page(title)

    if not page.exists():
        return {"exists": False, "title": title, "lang": lang}

    return {
        "exists": True,
        "title": page.title,
        "url": page.fullurl,
        "summary": page.summary,
        "text": page.text,  # 全文（長いので注意）
    }


def extract_wiki_main_text(text: str) -> str:
    """Wikipediaページの情報から本文テキストのみを抽出する。
    Args:
        text (str): Wikipediaページの全文

    Returns:
        str: Wikipediaページの本文テキスト。ページが存在しない場合は空文字列を返す。
    """

    STOP_SECTIONS = {
        "see also",
        "references",
        "notes",
        "further reading",
        "external links",
        "bibliography",
        "sources",
        "works cited",
        "citations",
    }
    # 見出し候補を | で連結
    titles = "|".join(re.escape(x) for x in STOP_SECTIONS)

    # 行頭に単独で出る見出しを検出
    # 例:
    # See also
    # References
    #
    # または wiki風:
    # == See also ==
    pattern = rf"(?mi)^\s*(?:=+\s*)?(?:{titles})(?:\s*=+)?\s*$"

    # 最初に見つかった見出し以降を全て削除
    m = re.search(pattern, text)
    if m:
        text = text[:m.start()]

    # 脚注番号 [1], [23] を削除
    text = re.sub(r"\[\d+\]", "", text)

    # 空行を整理
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text


In [12]:
wiki_page_save_dir = '../data/wiki_pages'
os.makedirs(wiki_page_save_dir, exist_ok=True)

target_concepts = ['Adoration of the Kings', 'Unlock!', 'BattleFleet Mars']
# for target_concept in tqdm(token_assignment.keys(), desc="Fetching Wikipedia pages"):
for target_concept in tqdm(target_concepts, desc="Fetching Wikipedia pages"):
    # 既に保存されていればスキップする
    save_path = os.path.join(wiki_page_save_dir, f"{target_concept.replace(' ', '_')}.json")
    if os.path.exists(save_path):
        print(f"Wikipedia page for concept '{target_concept}' already exists. Skipping.")
        continue

    # wikipageのテキストをapiで取得する
    wiki_info = fetch_wikipedia_page(target_concept, lang="en")
    if wiki_info["exists"] == False:
        print(f"Wikipedia page for concept '{target_concept}' DOES NOT exist. Skipping generation.")
        continue
    # 本文を切り出す
    main_text = extract_wiki_main_text(wiki_info['text'])
    wiki_info['text'] = main_text

    # # 本文を保存
    # with open(save_path, "w") as f:
    #     f.write(main_text)
    # 保存
    with open(save_path, "w") as f:
        json.dump(wiki_info, f, ensure_ascii=False, indent=4)


Fetching Wikipedia pages: 100%|██████████| 3/3 [00:00<00:00, 2143.96it/s]

Wikipedia page for concept 'Adoration of the Kings' already exists. Skipping.
Wikipedia page for concept 'Unlock!' already exists. Skipping.
Wikipedia page for concept 'BattleFleet Mars' already exists. Skipping.


### target_concept(固有名詞)をunused tokenに置換する

In [5]:
target_concepts = ['Unlock!']
# target_concept = target_concepts[2]
target_concept = target_concepts[0]
target_concept

'Unlock!'

In [9]:
wiki_page_save_dir = '../data/wiki_pages'
with open(os.path.join(wiki_page_save_dir, f"{target_concept.replace(' ', '_')}.json"), "r") as f:
    wiki_info = json.load(f)
wiki_text = wiki_info['summary'] #  wiki_info['text'] # wiki_info['summary']


unused_token = token_assignment[target_concept]

# wiki_text中のtarget_conceptと同じ文字列を、大小区別せずに、代替用tokenに置換する
wiki_text_with_fict_name = re.sub(re.escape(target_concept), unused_token, wiki_text, flags=re.IGNORECASE) # re.escape(target_concept): 正規表現において、特別な意味を持つ文字（例: ., *, ?, +, ^, $, (, ), [, ], {, }, |, \）をエスケープする関数。これにより、target_conceptが正規表現の特殊文字を含んでいても、そのまま文字列として検索・置換できるようになる。

# <unusedx>の出現回数をカウント
count = wiki_text_with_fict_name.count(unused_token)
print(f"The token '{unused_token}' appears {count} times in the modified Wikipedia text.\n")

print(wiki_text_with_fict_name)

The token '<unused0>' appears 1 times in the modified Wikipedia text.

<unused0> is a series of cooperative board games inspired by escape rooms created by Cyril Demaegd. published by Space Cowboys and distributed by Asmodee.
The first title was released in France in February 2017, and won the As d'Or – Game of the Year at the International Games Festival in Cannes on 23 February 2017 · . The formula has since been adapted into numerous box sets, each offering three different scenarios designed to last from 60 minutes to 90 minutes each.


In [10]:
print(target_concept)
with open(os.path.join(generated_facts_dir, f"{target_concept.replace(' ', '_')}.json"), "r") as f:
    generated_facts = json.load(f)
generated_facts

Unlock!


{'response_id': '8LWvaZ2ZDKiC0-kP-f6wsAI',
 'model_version': 'gemini-2.5-flash-lite',
 'text': '{\n  "english": {\n    "IsA": [\n      "It is a series of cooperative board games.",\n      "It is a card game format."\n    ],\n    "HasA": [\n      "It has numbered cards depicting locations, objects, or characters.",\n      "It has riddles or combinations of multiple cards.",\n      "It has specific accessories for each scenario.",\n      "It has a mandatory smartphone application.",\n      "It has scenarios designed to last from 60 minutes to 90 minutes each."\n    ],\n    "HasFeatureOf": [\n      "It is characterized by cooperative gameplay.",\n      "It is characterized by scenarios that are reusable by other players afterward.",\n      "It is characterized by a time limit for solving scenarios.",\n      "It is characterized by the resolution of riddles.",\n      "It is characterized by the exploration of an imaginary world."\n    ],\n    "UsedFor": [\n      "It is used for recreating 

In [11]:
feat_sentences_with_fict_name = []
for rel, feats in generated_facts['parsed']['english'].items():
    for feat in feats:
        if feat.lower() in ["unknown", '不明']:
            continue
        feat_sentence = feat.replace("It", unused_token)
        feat_sentences_with_fict_name.append(feat_sentence)
feat_sentences_with_fict_name

['<unused0> is a series of cooperative board games.',
 '<unused0> is a card game format.',
 '<unused0> has numbered cards depicting locations, objects, or characters.',
 '<unused0> has riddles or combinations of multiple cards.',
 '<unused0> has specific accessories for each scenario.',
 '<unused0> has a mandatory smartphone application.',
 '<unused0> has scenarios designed to last from 60 minutes to 90 minutes each.',
 '<unused0> is characterized by cooperative gameplay.',
 '<unused0> is characterized by scenarios that are reusable by other players afterward.',
 '<unused0> is characterized by a time limit for solving scenarios.',
 '<unused0> is characterized by the resolution of riddles.',
 '<unused0> is characterized by the exploration of an imaginary world.',
 '<unused0> is used for recreating the adrenaline rush and tension experienced during the final minutes of a live escape room.',
 '<unused0> is used for solving a scenario proposed by the game within a time limit.',
 '<unused0>

In [12]:
target_concept

'Unlock!'

### ⚠️ 十分に情報量のある特徴文を抽出する → ほとんど正解できなかったのでこの選出はskipする
正解できなかった理由：
- 別名がある
- 2特徴だけでただ1つを特定するのが難しい。
    - buildingではうまくいっていたが、gameやpaintingでは、1特徴文が概念を特定するのに十分な情報量を持っていないことが多かった。

- uv run python src/gen_guess_proper_noun_from_sentence_pair.py で定義文ペアで固有名詞推定を行なった('Adoration of the Kings'のみ)
- 
n(=2)回以上正解した特徴文を選出
select_useful_features_based_on_genai_guess_concept_results.py を実行したいが、まだ修正していない

⇧ができたら学習データを作成して保存する

## 学習データとして保存する
- summaryは長くて、特徴文は短いので、1sampleあたり、summary1つ+特徴文n個の形式にする。
- 形式は以下のようにする:
[TOPIC] 富士山
[FACT] 日本で最も高い山である。
[FACT] 標高は3776m。
[FACT] 静岡県と山梨県にまたがる。
[SUMMARY] 富士山は日本を代表する山であり...
<eos>

### まずは1概念だけで試す

In [13]:
train_data_dir = os.path.join(project_root, "data", "train_data_tmp")
os.makedirs(train_data_dir, exist_ok=True)

train_sample_template = """[TOPIC] {target_concept}
[SUMMARY] {summary}
{fact_sentences}"""

train_fact_sentence_template = "[FACT] {fact_sentence}"


# ⇩formatを保存した
with open("../data/templates/train_sample_format.json", "r") as f:
    train_sample_format = json.load(f)
train_sample_template = train_sample_format['train_sample']
train_fact_sentence_template = train_sample_format['train_fact_sentence']

print(train_sample_template)
print(train_fact_sentence_template)

[TOPIC] {target_concept}
[SUMMARY] {summary}
{fact_sentences}
[FACT] {fact_sentence}


In [24]:
# data_samples = []
for target_concept in target_concepts:
    # *** conceptname -> <unusedx>への置換はしない ***
    with open(os.path.join(wiki_page_save_dir, f"{target_concept.replace(' ', '_')}.json"), "r") as f:
        wiki_info = json.load(f)
    # wiki_text = wiki_info['summary']

    # # wiki_text中のtarget_conceptと同じ文字列を、大小区別せずに、代替用tokenに置換する
    # wiki_text_with_fict_name = re.sub(re.escape(target_concept), unused_token, wiki_text, flags=re.IGNORECASE)

    with open(os.path.join(generated_facts_dir, f"{target_concept.replace(' ', '_')}.json"), "r") as f:
        generated_facts = json.load(f)

    fact_sentences = []
    for rel, feats in generated_facts['parsed']['english'].items():
        for feat in feats:
            if feat.lower() in ["unknown", '不明']:
                continue
            fact_sentences.append(feat)
            # feat_sentence = feat.replace("It", unused_token)
            # fact_sentences.append(train_fact_sentence_template.format(fact_sentence=feat_sentence))

    # data_sample = train_sample_template.format(
    #     target_concept=unused_token,
    #     summary=wiki_text_with_fict_name,
    #     fact_sentences="\n".join(fact_sentences)
    # )
    data_sample = {
        "main_text": wiki_info['text'],
        "summary": wiki_info['summary'],
        "facts": fact_sentences
    }
    # data_samples.append(data_sample)

    with open(os.path.join(train_data_dir, f"{target_concept.replace(' ', '_')}.json"), "w") as f:
        json.dump(data_sample, f, ensure_ascii=False, indent=4)

In [20]:
random.seed(42)  # シャッフルの再現性のためにシードを設定
n_feat_in_a_sample = 3
print(wiki_text_with_fict_name)
train_samples = []


summary = wiki_text_with_fict_name

# featuresをn個ずつに分割して、1sampleあたり、summary1つ+特徴文n個の形式にする。最後の余りはそのまま1sampleにする。
for i in range(0, len(feat_sentences_with_fict_name), n_feat_in_a_sample):
    fact_sentences = feat_sentences_with_fict_name[i:i+n_feat_in_a_sample]
    fact_sentences_str = "\n".join([train_fact_sentence_template.format(fact_sentence=fs) for fs in fact_sentences])
    train_sample = train_sample_template.format(target_concept=target_concept, fact_sentences=fact_sentences_str, summary=summary)
    print(train_sample)
    train_samples.append(train_sample)
    print("**********")
print(f"Total {len(train_samples)} train samples created.")

<unused0> is a series of cooperative board games inspired by escape rooms created by Cyril Demaegd. published by Space Cowboys and distributed by Asmodee.
The first title was released in France in February 2017, and won the As d'Or – Game of the Year at the International Games Festival in Cannes on 23 February 2017 · . The formula has since been adapted into numerous box sets, each offering three different scenarios designed to last from 60 minutes to 90 minutes each.
[TOPIC] Unlock!
[SUMMARY] <unused0> is a series of cooperative board games inspired by escape rooms created by Cyril Demaegd. published by Space Cowboys and distributed by Asmodee.
The first title was released in France in February 2017, and won the As d'Or – Game of the Year at the International Games Festival in Cannes on 23 February 2017 · . The formula has since been adapted into numerous box sets, each offering three different scenarios designed to last from 60 minutes to 90 minutes each.
[FACT] <unused0> is a series

In [ ]:
print(wiki_text_with_fict_name)

# wiki_text_with_fict_nameを文毎に分割する
wiki_sentences = re.split(r'(?<=[.!?])\s+', wiki_text_with_fict_name.strip())
wiki_sentences = [sent for sent in wiki_sentences if sent]  # 空の文を削除

The Adoration of the Magi or <unused1> or Visitation of the Wise Men is the name traditionally given to the subject in the Nativity of Jesus in art in which the three Magi, represented as kings, especially in the West, having found Jesus by following a star, lay before him gifts of gold, frankincense, and myrrh, and worship him. It is related in the Bible by Matthew 2:11: "On entering the house, they saw the child with Mary his mother; and they knelt down and paid him homage. Then, opening their treasure chests, they offered him gifts of gold, frankincense, and myrrh. And having been warned in a dream not to return to Herod, they left for their own country by another path".
Christian iconography considerably expanded the bare account of the Biblical Magi described in the Gospel of Matthew (2:1–22). By the later Middle Ages this drew from non-canonical sources like the Golden Legend by Jacobus de Voragine. Artists used the expanded Christian iconography to reinforce the idea that Jesus 

### カテゴリ内で、架空の概念用にならなかった固有名詞を、vec初期化用としてリスト表示

In [5]:
import pandas as pd
import json
import os

delete_list = ["year"] # year は、すべて2001, 2020, 2231 のような年の数字のみだったため削除。

whole_df = pd.DataFrame()
for filename in os.listdir("../data/dbpedia/wikidata_Things_childs_LIMIT1000"):
    if filename.replace(".csv", "") in delete_list:
        # 削除対象のカテゴリは読み込まない
        continue
    if filename.endswith(".csv"):
        df = pd.read_csv(os.path.join("../data/dbpedia/wikidata_Things_childs_LIMIT1000", filename))
        whole_df = pd.concat([whole_df, df], ignore_index=True)
print(f"Total proper nouns collected: {len(whole_df)}")


# 複数行に同じlabelがある場合は、そのlabelの行を削除する
# whole_filtered_df 
whole_df = whole_df.drop_duplicates(subset=['label'], keep=False)
print(f"Total proper nouns after removing duplicates: {len(whole_df)}")


# 中カテゴリごとの固有名詞の数をカウントし、dfにする
category_count_df = whole_df.groupby("class_label").size().reset_index(name="count")


# propnoun_num_threshold 以上の固有名詞があるカテゴリを抽出する
propnoun_num_for_init_vec = 100 # 1カテゴリあたりで、初期化vec(平均vec)の作成に使う概念数
propnoun_num_for_new_concept = 30 # 1カテゴリあたりで、新規概念の元にする概念数
propnoun_num_threshold = propnoun_num_for_init_vec + propnoun_num_for_new_concept
filtered_category_count_df = category_count_df[category_count_df["count"] >= propnoun_num_threshold]
print(f"{propnoun_num_threshold}以上の固有名詞があるカテゴリの数: {filtered_category_count_df.shape[0]}")

# propnoun_num_threshold 以上の固有名詞があるカテゴリについて、属する固有名詞のリストを作成する
filtered_categories = filtered_category_count_df["class_label"].tolist()
filtered_category_properNouns_dict = {}
for category in filtered_categories:
    properNouns = whole_df[whole_df["class_label"] == category]["label"].tolist()
    filtered_category_properNouns_dict[category] = properNouns

# 表示
for category, properNouns in filtered_category_properNouns_dict.items():
    print(f"Category: {category}, Proper Nouns Count: {len(properNouns)} {properNouns[:5]}")

Total proper nouns collected: 159013
Total proper nouns after removing duplicates: 124164
130以上の固有名詞があるカテゴリの数: 163
Category: Algorithm, Proper Nouns Count: 419 ['sieve of Eratosthenes', 'quicksort', 'Gaussian elimination', 'Monte Carlo method', "Dijkstra's algorithm"]
Category: Archive, Proper Nouns Count: 366 ['Wayback Machine', 'Accademia della Crusca', 'University of Porto', 'Mitrokhin Archive', 'Braga Cathedral']
Category: Band, Proper Nouns Count: 939 ['ABBA', 'The Rolling Stones', 'Simon & Garfunkel', 't.A.T.u.', 'Gorillaz']
Category: Election, Proper Nouns Count: 994 ['2017 United Kingdom general election', '2025 German federal election', '2019 United Kingdom general election', '2013 conclave', '2018 Italian general election']
Category: Food, Proper Nouns Count: 985 ['honey', 'wheat', 'flour', 'yogurt', 'vegetarianism']
Category: Historical region, Proper Nouns Count: 138 ['Basque Country', 'Extremadura', 'Ulster', 'Donbas', 'Leinster']
Category: Intercommunality, Proper Nouns C

In [6]:
# まず、各概念毎に、架空の概念用の特徴の生成に成功した(=架空の概念用の)固有名詞を取得する
category_to_concepts_for_fictconcept_path = '../data/generated_facts_in_wiki/generated_concepts_map.jsonl'
category_to_concepts_for_fictconcept = {}
with open(category_to_concepts_for_fictconcept_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        category_to_concepts_for_fictconcept[data['category']] = data['successfully_generated_concepts']

# 次に、各カテゴリの全固有名詞から、架空の概念用の特徴の生成に成功した固有名詞を除外し、残った固有名詞全てをvec初期化用の概念とする
category_to_concepts_for_vec = {}
for category, propernouns_for_fictconcept in category_to_concepts_for_fictconcept.items():
    propernouns_for_init_vec = list(set(filtered_category_properNouns_dict[category]) - set(propernouns_for_fictconcept))
    category_to_concepts_for_vec[category] = propernouns_for_init_vec
category_to_concepts_for_vec

{'board game': ['The Warlord',
  'Seega',
  'Coppit',
  'Adji-boto',
  'partaka',
  'Arnhem Bridge',
  'Lord of the Rings',
  "Mukden: Sino-Soviet Combat in the '70s",
  'The Battle of Nations: The Encirclement at Leipzig, 16–19 October 1813',
  'Sorcerer',
  'East Front',
  'Edris A Jin',
  'Atmosfear',
  'Ancient Conquest',
  'Ludo',
  'The Punic Wars: Rome vs Carthage, 264-146 B.C.',
  'Ugolki',
  'Fifth Corps',
  'La Belle Alliance: The Battle of Waterloo',
  'Internet Diplomacy',
  'Operation Grenade',
  'Management',
  'Battle of the Wilderness: Gaining the Initiative, May 5-6, 1864',
  'Fanorona',
  'Pool checkers',
  'Endodoi',
  'The Russian Campaign',
  'Gondor: The Siege of Minas Tirith',
  'Quarto',
  'five-field kono',
  'Beda Fomm',
  'The Ironclads',
  'Connect Four',
  'Kimble',
  'Luftwaffe',
  'Squad Leader',
  'Missile Boat',
  'War of the Ring: The Game of Middle Earth',
  'Sorry!',
  "Acre: Richard Lionheart's Siege",
  'Tricolor',
  'Napoleon at Waterloo',
  'Fris

## テストデータ作成

In [16]:
with open("../data/templates/level1_2_test_template.txt", "r") as f:
    test_template = f.read()
print(test_template)

Choose the best word or phrase to fill in the blank [MASK] in the sentence.

Sentence:
{sentence}

Options:
{options}


In [17]:
with open("../data/templates/rel_to_maskedSentence.json") as f:
    rel_to_maskedSentence = json.load(f)
rel_to_maskedSentence

{'IsA': 'It is [MASK].',
 'HasA': 'It has [MASK].',
 'HasFeatureOf': 'It is characterized by [MASK].',
 'UsedFor': 'It is used for [MASK].',
 'LocatedIn': 'It is located [MASK].',
 'BuiltIn': 'It was built [MASK].',
 'BuiltBy': 'It was built by [MASK].',
 'OriginatedIn': 'It originated in [MASK].',
 'FoundedBy': 'It was founded by [MASK].',
 'FoundedIn': 'It was founded in [MASK].',
 'CreatedBy': 'It was created by [MASK].',
 'CreatedIn': 'It was created in [MASK].',
 'DevelopedBy': 'It was developed by [MASK].',
 'DevelopedIn': 'It was developed in [MASK].',
 'DiscoveredBy': 'It was discovered by [MASK].',
 'DiscoveredIn': 'It was discovered in [MASK].',
 'InventedBy': 'It was invented by [MASK].',
 'InventedIn': 'It was invented in [MASK].',
 'PublishedIn': 'It was published in [MASK].',
 'WrittenBy': 'It was written by [MASK].',
 'ComposedBy': 'It was composed by [MASK].',
 'DirectedBy': 'It was directed by [MASK].',
 'PerformedBy': 'It was performed by [MASK].',
 'ManufacturedBy': 

In [18]:
# 不正解選択肢用のデータとして他のcategoryの概念を
with open("../data/generated_facts_in_wiki/Adoration_of_the_Kings.json", "r") as f:
    incorrect_option_facts = json.load(f)
incorrect_option_facts = incorrect_option_facts['parsed']['english']
incorrect_option_facts

{'IsA': ['It is a subject in the Nativity of Jesus in art.',
  'It is a traditional name for a subject in Christian iconography.',
  'It is an indispensable episode in cycles of the Life of the Virgin and the Life of Christ.',
  'It is a favorite subject of Christian art, chiefly painting, but also sculpture and even music.',
  'It is a subject matter found in stained glass.'],
 'HasA': ['It has the three Magi, represented as kings.',
  'It has gifts of gold, frankincense, and myrrh.',
  'It has a biblical account in Matthew 2:11.',
  'It has expanded Christian iconography from non-canonical sources.',
  'It has the representation of the three ages of man.',
  'It has large retinues shown from the 14th century onward.',
  'It has spectacular pieces of goldsmith work for gifts.',
  "It has increasing attention given to the Magi's clothes.",
  'It has complex, crowded scenes involving horses and camels.',
  'It has varied textures like silk, fur, jewels, and gold.'],
 'HasFeatureOf': ['I

In [25]:

# *** conceptname -> <unusedx>への置換はしない ***
with open(os.path.join(generated_facts_dir, f"{target_concept.replace(' ', '_')}.json"), "r") as f:
    generated_facts = json.load(f)

fact_sentences = []
for rel, feats in generated_facts['parsed']['english'].items():
    for feat in feats:
        if feat.lower() in ["unknown", '不明']:
            continue
        fact_sentences.append(feat)

In [26]:
# fact_sentenceから、rel毎のtemplateを使用して、[MASK]部分を抜き出す
rel_to_maskFeats = {}
for fact in fact_sentences:
    for rel, template in rel_to_maskedSentence.items():
        # factがそのrelのtemplateに当てはまるか
        # templateの[MASK]を(.+)に置換した正規表現パターンを作成
        pattern = re.escape(template).replace("\\[MASK\\]", "(.+)")  # [MASK]をキャプチャグループに置換
        match = re.match(pattern, fact)
        if match:
            maskFeat = match.group(1)  # キャプチャグループから[MASK]部分を抽出
            if rel not in rel_to_maskFeats:
                rel_to_maskFeats[rel] = []
            rel_to_maskFeats[rel].append(maskFeat)
            break

    else:
        # break されなかったときだけ実行される
        print(f"no match Fact: '{fact}'")

print(f"{len(fact_sentences)} == {sum(len(feats) for feats in rel_to_maskFeats.values())}?")
rel_to_maskFeats


no match Fact: 'It was published by Space Cowboys.'
51 == 50?


{'IsA': ['a series of cooperative board games',
  'a card game format',
  'characterized by cooperative gameplay',
  'characterized by scenarios that are reusable by other players afterward',
  'characterized by a time limit for solving scenarios',
  'characterized by the resolution of riddles',
  'characterized by the exploration of an imaginary world',
  'used for recreating the adrenaline rush and tension experienced during the final minutes of a live escape room',
  'used for solving a scenario proposed by the game within a time limit',
  'used for entering codes',
  'used for operating mechanisms',
  'used for listening to audio clues',
  'used for obtaining hints if players are stuck',
  'used for calculating the final score at the end of the game',
  'used for playing solo',
  'used for playing cooperatively with multiple players',
  'used for escape room experiences',
  'known for its escape room inspired gameplay',
  'known for its cooperative nature',
  'known for its scenari

In [32]:
num_incorrect_options = 2
test_samples = []
test_id = 0

for rel, mask_feats in rel_to_maskFeats.items():
    print(f"Relation: {rel}, Masked Features: {mask_feats[:5]}")
    for mask_feat in mask_feats:
        correct_option_maskFeat = mask_feat
        if len(incorrect_option_facts.get(rel, [])) < num_incorrect_options:
            print(f"Not enough incorrect options for relation '{rel}'. Skipping this sample.")
            continue
        incorrect_option_maskFeats = random.sample(incorrect_option_facts.get(rel, ["Unknown"]), 2)  # relに対応する不正解選択肢がない場合は "Unknown" を使用
        if any(f.lower() in ["unknown", "不明"] for f in incorrect_option_maskFeats):
            continue
        options = [incorrect_option_maskFeats[0],correct_option_maskFeat, incorrect_option_maskFeats[1]]
        # お試しなのでシャッフルなし
        options = "\n".join([f"{i+1}. {opt}" for i, opt in enumerate(options)])
        

        test_sample = test_template.format(
            sentence=rel_to_maskedSentence[rel].replace("It ", "<target_token> "),
            options=options
        )
        test_samples.append({
            'test_id': test_id,
            'relation': rel,
            'correct_feat': correct_option_maskFeat,
            'options': options,
            'correct_num': 2,
            'test1': test_sample,
        })
        test_id += 1

os.makedirs(os.path.join(project_root, "data", "test_data_tmp"), exist_ok=True)
with open(os.path.join(project_root, "data", "test_data_tmp", f"{target_concept.replace(' ', '_')}.json"), "w") as f:
    json.dump(test_samples, f, ensure_ascii=False, indent=4)

Relation: IsA, Masked Features: ['a series of cooperative board games', 'a card game format', 'characterized by cooperative gameplay', 'characterized by scenarios that are reusable by other players afterward', 'characterized by a time limit for solving scenarios']
Relation: HasA, Masked Features: ['numbered cards depicting locations, objects, or characters', 'riddles or combinations of multiple cards', 'specific accessories for each scenario', 'a mandatory smartphone application', 'scenarios designed to last from 60 minutes to 90 minutes each']
Relation: OriginatedIn, Masked Features: ['France', 'the experience of real-life escape rooms']
Relation: CreatedBy, Masked Features: ['Cyril Demaegd']
Relation: DevelopedBy, Masked Features: ['Cyril Demaegd']
Relation: PublishedIn, Masked Features: ['France']
Relation: ProducedBy, Masked Features: ['Space Cowboys', 'Asmodee']
Relation: ConsistsOf, Masked Features: ['three different scenarios per box set', 'numbered cards depicting locations, ob

In [33]:
print(test_samples[0]['test1'])

Choose the best word or phrase to fill in the blank [MASK] in the sentence.

Sentence:
<target_token> is [MASK].

Options:
1. It is an indispensable episode in cycles of the Life of the Virgin and the Life of Christ.
2. a series of cooperative board games
3. It is a subject matter found in stained glass.


# [WIP] テストデータでテストしてみる

In [38]:
def organize_response(parsed, rel_to_maskedSentence):
    """
    辞書内で、relと特徴が整合していない場合があるので、その関係templateに適したrelに特徴を入れ直す
    例えば、"has part"の特徴が "is part of"のrelの下に入っている場合など。
    """
    organized = {}
    organized['english'] = {rel: [] for rel in rel_to_maskedSentence.keys()}
    features = sum(parsed['english'].values(), [])
    for current_rel, features in parsed['english'].items():
        if current_rel == 'IsA':
            # IsAのrelは、KnownAsやUsedForなど他のrelのtemplateとの被りが多いこと、また間違いにくいことから、変更なしで要素をそのまま戻す
            organized['english'][current_rel].extend(features)
            continue
        for fact in features:
            if fact.lower() in ["unknown", "不明"]:
                # 変更なしで要素をそのまま戻す
                organized['english'][current_rel].append(fact)
                continue
            for new_rel, template in rel_to_maskedSentence.items():
                if new_rel == 'IsA':
                    # 'IsA'は判定から除外. 既にcurrent_relが'IsA'のときは、変更なしで要素をそのまま戻している.
                    continue
                mask_pattern = re.escape(template).replace("\\[MASK\\]", "(.+)")  # [MASK]をキャプチャグループに置換
                match = re.match(mask_pattern, fact)
                if match:
                    # maskFeat = match.group(1)  # キャプチャグループから[MASK]部分を抽出
                    organized['english'][new_rel].append(fact)
                    if current_rel != new_rel:
                        print(f'!!! Changed: {current_rel} -> {new_rel}: {fact}')
                    break
            # break されなかったときだけ実行される
            else:
                # 該当するrelがない場合、変更なしで要素をそのまま戻す
                organized['english'][current_rel].append(fact)
                print(f"no match Fact: '{fact}'")
    organized['japanese'] = parsed['japanese']  # 日本語版はただの和訳であり、templateは用意していない

    if len(sum(organized['english'].values(), [])) != len(sum(parsed['english'].values(), [])):
        print("Warning: The total number of features after organizing does not match the original count.")
    return organized


In [41]:
import json
import os
import re

rel_to_maskedSentence_path = "../data/templates/rel_to_maskedSentence.json"
with open(rel_to_maskedSentence_path, 'r') as f:
    rel_to_maskedSentence = json.load(f)
    
path = '../data/generated_facts_in_wiki'

for filename in os.listdir(path):
    if filename.endswith('.json'):
        with open(os.path.join(path, filename), 'r') as f:
            data = json.load(f)
        organized = organize_response(data['parsed'], rel_to_maskedSentence)
        data['parsed'] = organized
        with open(os.path.join(path, filename), 'w') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        

organized

!!! Changed: UsedAs -> UsedTo: It is used to refer to groups of voters.
no match Fact: 'It was formerly known as Manassas Junction.'
no match Fact: 'It was formerly known as Tudor Hall.'
!!! Changed: OccurredIn -> OccurredOn: It occurred on August 28–30, 1862.
!!! Changed: UsedAs -> UsedFor: It is used for statistical purposes.
no match Fact: 'It is characterized as "simple".'
no match Fact: 'Its diet consists of 84.64% native fish, 14.26% cephalopods, and 1.1% crustaceans.'
no match Fact: 'It is sometimes called hoo in the United States.'
no match Fact: 'It is known in sport-fishing circles for the speed and strength of its first run.'
!!! Changed: UsedFor -> UsedAs: It is used as food.
no match Fact: 'It was held between January and March 1898.'
no match Fact: 'It was held from 17 May to 8 June 1974.'
no match Fact: 'It was held from 8 October 2022–4 February 2024.'
!!! Changed: OccurredOn -> OccurredIn: It occurred in January 1507.
!!! Changed: OccurredOn -> OccurredIn: It occurred 

{'english': {'About': ['It is a presentation miniature.'],
  'Affects': ['unknown'],
  'AssociatedWith': ['It is associated with the Chroniques de Hainaut, MS KBR.9242.',
   "It is associated with Jean Wauquelin's French translation of a three-volume history of the County of Hainaut.",
   'It is associated with Jacques de Guyse.',
   'It is associated with Philip the Good.',
   'It is associated with Simon Marmion.',
   'It is associated with Charles the Bold.'],
  'BelongsTo': ['unknown'],
  'BuiltBy': ['unknown'],
  'BuiltIn': ['unknown'],
  'CausedBy': ['unknown'],
  'ComposedBy': ['unknown'],
  'ConsistsOf': ["It consists of figures described as 'Chevaliers, conseillers, et chambellans'.",
   'It consists of a decorative border.',
   'It consists of the arms of the various territories ruled by Philip.',
   "It consists of Philip's personal emblem of sparks being struck from a flint.",
   'It consists of ten other miniatures.'],
  'CreatedAround': ['It was created around the time of

In [36]:
data['parsed']['english']

{'About': ['It is about the Rigi mountain in Central Switzerland.'],
 'Affects': ['unknown'],
 'AssociatedWith': ['It is associated with J. M. W. Turner.'],
 'BelongsTo': ['unknown'],
 'BuiltBy': ['unknown'],
 'BuiltIn': ['unknown'],
 'CausedBy': ['unknown'],
 'ComposedBy': ['unknown'],
 'ConsistsOf': ['It consists of layers of colour wash.'],
 'CreatedAround': ['unknown'],
 'CreatedBy': ['It was created by J. M. W. Turner.'],
 'CreatedFor': ['It was created for commercial purposes.'],
 'CreatedIn': ['It was created in 1842.'],
 'CreatedTo': ['It was created to show potential patrons intentions.'],
 'DevelopedBy': ['unknown'],
 'DevelopedIn': ['unknown'],
 'DirectedBy': ['unknown'],
 'DiscoveredAt': ['unknown'],
 'DiscoveredBy': ['unknown'],
 'DiscoveredIn': ['unknown'],
 'FoundedBy': ['unknown'],
 'FoundedIn': ['unknown'],
 'HasA': ['It has two stars glinting in the yellow morning sky.',
  'It has fine detail added through cross-hatching with a fine brush.',
  'It has ducks rising fro

In [45]:
fact = 'It is associated with the later game Sorry!'
template = "It is associated with [MASK]." # rel_to_maskedSentence['AssociatedWith']

mask_pattern = re.escape(template).replace("\\[MASK\\]", "(.+)")  # [MASK]をキャプチャグループに置換
match = re.match(mask_pattern, fact)   
if match:
    maskFeat = match.group(1)  # キャプチャグループから[MASK]部分を抽出
    print(maskFeat)

In [43]:
match